# M3 · Prévision du rendement — entraînement

**Tâche** : estimer la production d'olives de la saison, puis la récolte d'une parcelle.

**Plan B du plan de projet** : les productions par gouvernorat (rapports ONAGRI en PDF) ne sont pas disponibles
en données ouvertes ; on prévoit donc la **production nationale** de la Tunisie, puis le rendement par hectare et
par arbre appliqué à la parcelle.

| | |
|---|---|
| Données | **FAOSTAT** (Tunisie, olives : production, surface récoltée, rendement, 1961 → dernière année publiée) · météo **ERA5** journalière (Open-Meteo) moyennée sur 12 gouvernorats oléicoles |
| Variables | production des 2 saisons précédentes (alternance), tendance, pluie septembre → mai, pluie d'été, jours ≥ 35 °C en avril-mai (floraison), Tmax moyenne de l'été |
| Validation | **chronologique glissante** : chaque saison du test est prévue par un modèle entraîné uniquement sur les saisons précédentes |
| Baseline | régression linéaire (le plan) ; aussi « comme la saison précédente » |
| Modèles | Ridge, Random Forest, Gradient Boosting |
| Métriques | MAE (t), MAPE (%) |
| Export | `models/m3/model.joblib`, `history.csv`, `metadata.json` |

In [ ]:
%matplotlib inline
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

import m3_common as mc

np.random.seed(mc.SEED)
mc.OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.precision", 1)
FIRST_TEST_SEASON = 2005
print("python", platform.python_version(), "| scikit-learn", sklearn.__version__)

## 1 · Données

In [ ]:
fao = mc.load_faostat()
weather_daily = mc.load_weather()
weather = mc.seasonal_weather(weather_daily)
data = mc.build_dataset(fao, weather)
print(f"FAOSTAT : {fao.index.min()} → {fao.index.max()} ({len(fao)} saisons)")
print(f"météo   : saisons {weather.index.min()} → {weather.index.max()}")
print(f"jeu d'apprentissage : {len(data)} saisons ({data.index.min()} → {data.index.max()})")
fao.tail(6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
(fao["production_t"] / 1e6).plot(ax=axes[0], marker=".", color="tab:green")
axes[0].set(title="Production d'olives en Tunisie (millions de t, FAOSTAT)", xlabel="saison")
axes[1].scatter(data["rain_sep_may"], data["production_t"] / 1e6, c=data.index, cmap="viridis")
axes[1].set(title="Production vs pluie septembre → mai", xlabel="pluie (mm, moyenne 12 gouvernorats)", ylabel="Mt")
fig.tight_layout()
fig.savefig(mc.OUTPUT_DIR / "eda.png", dpi=110)
plt.show()
data[mc.FEATURES + ["production_t"]].corr()["production_t"].drop("production_t").to_frame("corrélation avec la production")

**Lecture** : la production tunisienne alterne très fortement d'une année sur l'autre (« bonne » et « mauvaise »
années de l'olivier), avec une tendance à la hausse (nouvelles plantations) et une forte dépendance à la pluie.
Les saisons sèches (2022–2023) se voient nettement.

## 2 · Prévisions glissantes (saisons 2005 → dernière connue)

In [ ]:
actual = data.loc[data.index >= FIRST_TEST_SEASON, "production_t"]
preds = {"Comme la saison précédente": np.exp(data.loc[actual.index, "lag1_log"])}
for name in mc.models():
    preds[name] = mc.rolling_forecasts(data, lambda n=name: mc.models()[n], FIRST_TEST_SEASON)
results = pd.DataFrame({name: mc.metrics(actual, p.reindex(actual.index)) for name, p in preds.items()}).T
results = results.sort_values("MAPE (%)")
results.to_csv(mc.OUTPUT_DIR / "results.csv")
results

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(actual.index, actual / 1e6, "k-o", label="FAOSTAT (réel)")
for name in results.index[:3]:
    ax.plot(actual.index, preds[name].reindex(actual.index) / 1e6, "--", marker=".", label=name)
ax.set(title="Production prévue un an à l'avance (Mt)", xlabel="saison")
ax.legend()
fig.tight_layout()
fig.savefig(mc.OUTPUT_DIR / "rolling_forecasts.png", dpi=110)
plt.show()

## 3 · Choix du modèle, incertitude et export

On garde le modèle à la plus petite MAPE (hors « comme la saison précédente », qui n'est pas un modèle). L'intervalle
de la parcelle vient des **erreurs réelles du test glissant** (quantiles 10 % et 90 % de l'erreur relative).

In [ ]:
candidates = results.drop(index="Comme la saison précédente")
best = candidates.index[0]
print("modèle retenu :", best)
rel_err = (preds[best].reindex(actual.index) - actual) / actual
q10, q90 = float(rel_err.quantile(0.10)), float(rel_err.quantile(0.90))
print(f"erreur relative du test : 10 % = {q10:+.0%}, 90 % = {q90:+.0%}")

final = mc.models()[best]
final.fit(data[mc.FEATURES], data[mc.TARGET])
if hasattr(final, "feature_importances_"):
    importance = pd.Series(final.feature_importances_, index=mc.FEATURES).sort_values()
    importance.plot.barh(figsize=(6, 3), title="Importance des variables")
    plt.tight_layout()
    plt.savefig(mc.OUTPUT_DIR / "importance.png", dpi=110)
    plt.show()

history = fao.join(weather, how="outer")
history.index.name = "season"
metadata = {
    "model": best,
    "scope": "NATIONAL",
    "features": mc.FEATURES,
    "relative_error_quantiles": {"q10": round(q10, 4), "q90": round(q90, 4)},
    "data": {
        "faostat": [int(fao.index.min()), int(fao.index.max())],
        "weather_seasons": [int(weather.index.min()), int(weather.index.max())],
        "regions": list(mc.REGIONS),
        "first_test_season": FIRST_TEST_SEASON,
    },
    "metrics_rolling_test": json.loads(results.round(3).to_json(orient="index")),
    # Tunisia: about 100 million olive trees (ONH / ministère de l'Agriculture), used only when the parcel
    # has a tree count but no area: kg per tree = national yield per ha / national density.
    "national_trees": 100_000_000,
    "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__, "numpy": np.__version__},
}
mc.export({"model.joblib": final, "history.csv": history}, metadata)